In [1]:
# =================================================================
# COMMENTARY: 
# Questa cella carica i dataset dalla cartella "Data" e li fonde.
# Crea il vettore Target (Y): 1 se Carlsen vince, 0 altrimenti.
# NOTA: Corretto il nome della colonna da 'ply' a 'move_no' per 
# compatibilità con il file specifico di Magnus Carlsen.
# =================================================================

import pandas as pd
import numpy as np
import chess

# 1. Caricamento dei Dati dalla cartella "Data"
print("Caricamento dataset in corso...")
df_info = pd.read_csv('./Data/Carlsen_game_info.csv')
df_moves = pd.read_csv('./Data/Carlsen_moves.csv') 

# 2. Creazione del Target Matematico (Y)
df_info['Target_Carlsen'] = df_info['winner'].apply(lambda x: 1 if x == 'Carlsen,M' or x == 'Carlsen,Magnus' else 0)
df_info_clean = df_info[['game_id', 'white', 'black', 'Target_Carlsen']]

# 3. Fusione (Merge) per associare il Target a ogni singola FEN
df_completo = pd.merge(df_moves, df_info_clean, on='game_id', how='inner')

# 4. Filtro per lo sviluppo (Campione di test per evitare saturazione RAM)
id_campione = df_completo['game_id'].unique()[:5]
df_sample = df_completo[df_completo['game_id'].isin(id_campione)].copy()

print(f"Dataset fuso con successo! Dimensione totale: {len(df_completo)} posizioni.")
print(f"Lavoreremo su un campione di test di {len(df_sample)} posizioni.")

# 5. Stampa a schermo delle colonne corrette
display(df_sample[['game_id', 'move_no', 'fen', 'Target_Carlsen']].head())

Caricamento dataset in corso...
Dataset fuso con successo! Dimensione totale: 396129 posizioni.
Lavoreremo su un campione di test di 274 posizioni.


,game_id,move_no,fen,Target_Carlsen
0,0a17b497-1f25-42ea-969f-60b853ed08a2,1,rnbqkbnr/pppppppp/8/8/3P4/8/PPP1PPPP/RNBQKBNR,0
1,0a17b497-1f25-42ea-969f-60b853ed08a2,2,rnbqkb1r/pppppppp/5n2/8/3P4/8/PPP1PPPP/RNBQKBNR,0
2,0a17b497-1f25-42ea-969f-60b853ed08a2,3,rnbqkb1r/pppppppp/5n2/8/3P4/5N2/PPP1PPPP/RNBQKB1R,0
3,0a17b497-1f25-42ea-969f-60b853ed08a2,4,rnbqkb1r/ppp1pppp/5n2/3p4/3P4/5N2/PPP1PPPP/RNB...,0
4,0a17b497-1f25-42ea-969f-60b853ed08a2,5,rnbqkb1r/ppp1pppp/5n2/3p4/3P4/4PN2/PPP2PPP/RNB...,0


In [2]:
# =================================================================
# COMMENTARY:
# Questa cella definisce la funzione di Tapering continuo (p1, p2, p3).
# Basato sul materiale non-pedonale (Max 62):
# - Apertura (p1): inizia a sfumare sotto 55 punti e si azzera a 40.
# - Finale (p3): inizia a crescere sotto 30 punti e domina sotto 15.
# - Mediogioco (p2): prende sempre la percentuale rimanente (1 - p1 - p3), 
#   avendo il suo picco assoluto (1.0) nella fascia centrale tra 30 e 40.
# Questo garantisce transizioni matematicamente continue e lisce.
# =================================================================

import chess

def calcola_pesi_fasi(board):
    """
    Ritorna una tupla continua (p_apertura, p_mediogioco, p_finale) 
    in base al materiale non pedonale. La somma è sempre 1.0.
    """
    valori_pezzi = {chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9}
    materiale_totale = 0
    
    for pezzo_tipo, valore in valori_pezzi.items():
        materiale_totale += len(board.pieces(pezzo_tipo, chess.WHITE)) * valore
        materiale_totale += len(board.pieces(pezzo_tipo, chess.BLACK)) * valore

    # 1. Calcolo Punteggio Apertura (Interpolazione lineare tra 40 e 55)
    if materiale_totale >= 55:
        p_apertura = 1.0
    elif materiale_totale <= 40:
        p_apertura = 0.0
    else:
        p_apertura = (materiale_totale - 40) / 15.0

    # 2. Calcolo Punteggio Finale (Interpolazione lineare tra 15 e 30)
    if materiale_totale <= 15:
        p_finale = 1.0
    elif materiale_totale >= 30:
        p_finale = 0.0
    else:
        p_finale = (30 - materiale_totale) / 15.0

    # 3. Calcolo Punteggio Mediogioco (Per differenza)
    p_mediogioco = 1.0 - p_apertura - p_finale
    
    # Clip per sicurezza numerica contro gli arrotondamenti di Python
    p_mediogioco = max(0.0, min(1.0, p_mediogioco))

    return round(p_apertura, 4), round(p_mediogioco, 4), round(p_finale, 4)

# --- TEST DI COLLAUDO ---
print("Test Tapering Continuo:")

# 1. Posizione iniziale (Materiale = 62)
b_inizio = chess.Board()
print(f"Inizio (Mat=62): {calcola_pesi_fasi(b_inizio)} -> 100% Apertura")

# 2. Le Regine e un pezzo leggero scambiati (Materiale = 38)
# Rientra perfettamente nella fascia 30-40
b_media = chess.Board("r1b1r1k1/pp3ppp/2n2n2/3p4/3P4/2N2N2/PP2BPPP/R3R1K1 w - - 0 15")
print(f"Medio  (Mat=38): {calcola_pesi_fasi(b_media)} -> 100% Mediogioco")

# 3. Posizione molto semplificata, si entra nel finale (Materiale = 26)
b_trans = chess.Board("r3r1k1/pp3ppp/5n2/3p4/3P4/5N2/PP3PPP/R3R1K1 w - - 0 20")
print(f"Trans. (Mat=26): {calcola_pesi_fasi(b_trans)} -> Sfumatura tra Medio e Finale")

# 4. Re e Torre contro Re (Materiale = 5)
b_finale = chess.Board("8/8/8/4k3/8/4K3/8/2R5 w - - 0 1")
print(f"Fine   (Mat=5) : {calcola_pesi_fasi(b_finale)} -> 100% Finale")

Test Tapering Continuo:
Inizio (Mat=62): (1.0, 0.0, 0.0) -> 100% Apertura
Medio  (Mat=38): (0.0, 1.0, 0.0) -> 100% Mediogioco
Trans. (Mat=26): (0.0, 0.7333, 0.2667) -> Sfumatura tra Medio e Finale
Fine   (Mat=5) : (0.0, 0.0, 1.0) -> 100% Finale


In [3]:
# =================================================================
# COMMENTARY:
# Costruzione della Giga-Matrice X e del vettore Y.
# Il vettore base ora conta 768 feature spaziali + 14 feature euristiche avanzate (Totale 782).
# Le 14 euristiche coprono: Materiale, Centro, Sviluppo, Alfieri, Torri (7a trav. e Colonne Aperte),
# Sicurezza Re (Arrocco e Scudo), Struttura Pedonale (Passati, Doppiati, Isolati),
# Mobilità, Spazio, Iniziativa e Attività del Re nel finale.
# Il Super-Vettore finale, unendo le 3 fasi, avrà 782 * 3 = 2346 feature!
# =================================================================

import numpy as np
import chess
import time

def estrai_posizioni_spaziali(board):
    """Estrae le 768 posizioni geometriche binarie (12 blocchi da 64 case)"""
    features = np.zeros(768, dtype=np.int8)
    for color in [chess.WHITE, chess.BLACK]:
        color_offset = 0 if color == chess.WHITE else 6
        for piece_type in range(1, 7): 
            squares = board.pieces(piece_type, color)
            block_idx = color_offset + (piece_type - 1)
            for sq in squares:
                features[(block_idx * 64) + sq] = 1
    return features

def estrai_euristiche_avanzate(board):
    """
    Calcola 14 euristiche scacchistiche statiche.
    La formula è sempre: (Valore Bianco) - (Valore Nero).
    Risultato > 0 significa vantaggio per il Bianco.
    """
    e = []
    
    # 1. MATERIALE (Il classico)
    valori = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9}
    mat_w = sum(len(board.pieces(pt, chess.WHITE)) * val for pt, val in valori.items())
    mat_b = sum(len(board.pieces(pt, chess.BLACK)) * val for pt, val in valori.items())
    e.append(mat_w - mat_b)
    
    # 2. CENTRO (Controllo d4, e4, d5, e5)
    centro = [chess.D4, chess.E4, chess.D5, chess.E5]
    centro_w = sum(1 for sq in centro if board.color_at(sq) == chess.WHITE)
    centro_b = sum(1 for sq in centro if board.color_at(sq) == chess.BLACK)
    e.append(centro_w - centro_b)
    
    # 3. SVILUPPO (Pezzi leggeri fuori dalla prima traversa)
    sviluppo_w = 4 - sum(1 for sq in [chess.B1, chess.C1, chess.F1, chess.G1] if board.color_at(sq) == chess.WHITE)
    sviluppo_b = 4 - sum(1 for sq in [chess.B8, chess.C8, chess.F8, chess.G8] if board.color_at(sq) == chess.BLACK)
    e.append(sviluppo_w - sviluppo_b)
    
    # 4. COPPIA DEGLI ALFIERI
    alf_w = 1 if len(board.pieces(chess.BISHOP, chess.WHITE)) >= 2 else 0
    alf_b = 1 if len(board.pieces(chess.BISHOP, chess.BLACK)) >= 2 else 0
    e.append(alf_w - alf_b)
    
    # 5. TORRI IN 7a TRAVERSA (2a per il Nero)
    t7_w = sum(1 for sq in board.pieces(chess.ROOK, chess.WHITE) if chess.square_rank(sq) == 6)
    t7_b = sum(1 for sq in board.pieces(chess.ROOK, chess.BLACK) if chess.square_rank(sq) == 1)
    e.append(t7_w - t7_b)

    # 6. TORRI SU COLONNE APERTE (Nessun pedone sulla colonna)
    colonne_pedoni = set(chess.square_file(sq) for sq in board.pieces(chess.PAWN, chess.WHITE) | board.pieces(chess.PAWN, chess.BLACK))
    ta_w = sum(1 for sq in board.pieces(chess.ROOK, chess.WHITE) if chess.square_file(sq) not in colonne_pedoni)
    ta_b = sum(1 for sq in board.pieces(chess.ROOK, chess.BLACK) if chess.square_file(sq) not in colonne_pedoni)
    e.append(ta_w - ta_b)
    
    # 7. DIRITTI DI ARROCCO E SICUREZZA
    sic_w = 1 if board.has_castling_rights(chess.WHITE) else 0
    sic_b = 1 if board.has_castling_rights(chess.BLACK) else 0
    e.append(sic_w - sic_b)

    # 8. SCUDO PEDONALE DEL RE (Pedoni nelle vicinanze del proprio re)
    kw_file, kw_rank = chess.square_file(board.king(chess.WHITE)), chess.square_rank(board.king(chess.WHITE))
    kb_file, kb_rank = chess.square_file(board.king(chess.BLACK)), chess.square_rank(board.king(chess.BLACK))
    scudo_w = sum(1 for sq in board.pieces(chess.PAWN, chess.WHITE) if abs(chess.square_file(sq) - kw_file) <= 1 and chess.square_rank(sq) - kw_rank in [1, 2])
    scudo_b = sum(1 for sq in board.pieces(chess.PAWN, chess.BLACK) if abs(chess.square_file(sq) - kb_file) <= 1 and kb_rank - chess.square_rank(sq) in [1, 2])
    e.append(scudo_w - scudo_b)
    
    # 9. PEDONI DOPPIATI (Penalità: calcoliamo chi ne ha di più e invertiamo)
    file_w = [chess.square_file(sq) for sq in board.pieces(chess.PAWN, chess.WHITE)]
    file_b = [chess.square_file(sq) for sq in board.pieces(chess.PAWN, chess.BLACK)]
    doppiati_w = sum(1 for count in [file_w.count(f) for f in range(8)] if count > 1)
    doppiati_b = sum(1 for count in [file_b.count(f) for f in range(8)] if count > 1)
    e.append(doppiati_b - doppiati_w) # Vantaggio al bianco se il nero ha doppiati
    
    # 10. PEDONI ISOLATI (Pedoni senza compagni nelle colonne adiacenti)
    set_file_w = set(file_w)
    set_file_b = set(file_b)
    iso_w = sum(1 for f in file_w if (f-1) not in set_file_w and (f+1) not in set_file_w)
    iso_b = sum(1 for f in file_b if (f-1) not in set_file_b and (f+1) not in set_file_b)
    e.append(iso_b - iso_w) # Penalità scambiata

    # 11. MOBILITÀ (Numero di case attaccate / raggio d'azione)
    mob_w = sum(len(board.attacks(sq)) for sq in chess.SQUARES if board.color_at(sq) == chess.WHITE)
    mob_b = sum(len(board.attacks(sq)) for sq in chess.SQUARES if board.color_at(sq) == chess.BLACK)
    e.append((mob_w - mob_b) * 0.1) # Scalato per non sbilanciare la magnitudo
    
    # 12. VANTAGGIO DI SPAZIO (Pedoni avanzati oltre la 4a traversa)
    spazio_w = sum(1 for sq in board.pieces(chess.PAWN, chess.WHITE) if chess.square_rank(sq) >= 4)
    spazio_b = sum(1 for sq in board.pieces(chess.PAWN, chess.BLACK) if chess.square_rank(sq) <= 3)
    e.append(spazio_w - spazio_b)

    # 13. INIZIATIVA (Scacco in corso)
    iniziativa_w = 1 if board.is_check() and board.turn == chess.WHITE else 0
    iniziativa_b = 1 if board.is_check() and board.turn == chess.BLACK else 0
    e.append(iniziativa_w - iniziativa_b)

    # 14. ATTIVITÀ DEL RE NEL FINALE (Vicinanza al centro: casa d4/e4/d5/e5)
    # Calcolato come la distanza di Chebyshev dal centro. Più è bassa, meglio è.
    dist_w = max(abs(kw_file - 3.5), abs(kw_rank - 3.5))
    dist_b = max(abs(kb_file - 3.5), abs(kb_rank - 3.5))
    e.append(dist_b - dist_w) # Invertita: se il nero è più lontano (distanza maggiore), vantaggio bianco.

    return np.array(e, dtype=np.float32)

def estrai_vettore_base(board):
    """Fonde le 768 coordinate spaziali binarie con le 14 euristiche (Tot: 782)"""
    spaziali = estrai_posizioni_spaziali(board)
    strategiche = estrai_euristiche_avanzate(board)
    return np.concatenate([spaziali, strategiche])

# =================================================================
# FASE DI CALCOLO DELLA GIGA-MATRICE
# =================================================================
print(f"Avvio estrazione della Giga-Matrice per {len(df_sample)} posizioni...")
start_time = time.time()

X_list = []
y_list = []

for idx, row in df_sample.iterrows():
    fen = row['fen']
    target = row['Target_Carlsen']
    
    board = chess.Board(fen)
    
    # 1. Estraiamo il vettore base (782 feature)
    base_features = estrai_vettore_base(board)
    
    # 2. Moltiplicatori di Tapering (Dalla Cella 2)
    p1, p2, p3 = calcola_pesi_fasi(board)
    
    # 3. Creiamo il Super-Vettore (782 * 3 = 2346 elementi)
    feat_apertura = base_features * p1
    feat_mediogioco = base_features * p2
    feat_finale = base_features * p3
    
    super_vector = np.concatenate([feat_apertura, feat_mediogioco, feat_finale]).astype(np.float32)
    
    X_list.append(super_vector)
    y_list.append(target)

X_matrix = np.array(X_list)
y_vector = np.array(y_list, dtype=np.int8)

tempo_estrazione = time.time() - start_time

print("-" * 50)
print(f"✅ Estrazione completata in {tempo_estrazione:.2f} secondi!")
print(f"Forma della Matrice X: {X_matrix.shape} (Posizioni x Feature)")
print(f"Forma del Vettore Y:   {y_vector.shape}")
print("-" * 50)

# Decommentare per salvare i risultati compressi sul disco:
# np.savez_compressed('Data/X_y_carlsen_sample.npz', X=X_matrix, y=y_vector)
# print("Dati estratti e salvati in 'Data/X_y_carlsen_sample.npz'")

Avvio estrazione della Giga-Matrice per 274 posizioni...
--------------------------------------------------
✅ Estrazione completata in 0.09 secondi!
Forma della Matrice X: (274, 2346) (Posizioni x Feature)
Forma del Vettore Y:   (274,)
--------------------------------------------------
